# RAVDESS test accuracy by training allocation
Run 23 (a5v5av10) is the baseline. Run 24 (a1v9av10) moves four training actors from audio-only to video-only. Run 25 (a9v1av10) makes the reverse allocation change. Each allocation retains ten audiovisual training actors.

The notebook uses the first five **jointly completed** folds in numeric order. Each must have four completed blocks in every run. The configuration cell can select specific fold IDs or increase the count to six. Choose one prediction source for the whole comparison. `final_epoch` uses the original multi-head model’s final training-epoch test CSVs. `single_head_retest` uses the modified model’s three test CSVs and metrics file.

Bar heights and error bars show the mean and sample SD across the 20 fold/block models. Each emotion's accuracy is the percentage of its held-out trials classified correctly. Differences are paired by fold and block and expressed in percentage points. For exploratory two-sided paired t tests, we first average the four blocks within each fold because they share test actors. Training actors overlap across folds. P values are unadjusted across comparisons.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel
from IPython.display import display

ROOT = Path(r'C:/Users/sosc/Desktop/alex/visual_audio_emotion/HiCMAE_results')
RUN_LABELS = {23:'a5v5av10', 24:'a1v9av10', 25:'a9v1av10'}
BASELINE = 23
N_FOLDS = 5
FOLD_IDS = [1, 2, 3, 4, 5]  # Set to None to use the first N_FOLDS shared folds.
PREDICTION_SOURCE = 'final_epoch'  # Or 'single_head_retest' for the modified model.
BLOCKS = (1, 2, 3, 4)
comparisons = [r for r in RUN_LABELS if r != BASELINE]
RUNS = {r: ROOT / f'run{r}' for r in RUN_LABELS}
def offsets(n):
    return (np.arange(n) - (n-1)/2) * .65 / max(n,1)
condition_titles = {'audio_neutral':'Audio only', 'video_noise':'Video only', 'both':'Audiovisual'}

conditions = ['audio_neutral','video_noise','both']
emotions = ['happy','sad','angry','fearful','surprised','disgusted']
keys = ['fold','block','condition','actor','trial']


In [ ]:
configs = {r: json.loads((path / 'experiment.json').read_text()) for r, path in RUNS.items()}

if PREDICTION_SOURCE not in ('final_epoch', 'single_head_retest'):
    raise ValueError('Choose final_epoch or single_head_retest.')

def prediction_dir(run, fold, block):
    block_dir = RUNS[run] / f'fold_{fold:02d}' / f'block_{block:02d}'
    if PREDICTION_SOURCE == 'single_head_retest':
        return block_dir / 'single_head_retest'
    return block_dir / f"epoch_{configs[run]['epochs']:03d}"

def missing_file(run, fold):
    names = [f'test_{c}.csv' for c in conditions]
    if PREDICTION_SOURCE == 'single_head_retest':
        names.append('metrics.json')
    for block in BLOCKS:
        folder = prediction_dir(run, fold, block)
        for path in [folder.parent / 'test.csv', *(folder / name for name in names)]:
            if not path.is_file():
                return path
    return None

ready = {}
for r, cfg in configs.items():
    if 'a{}v{}av{}'.format(*cfg['train_actors']) != RUN_LABELS[r] or cfg['emotions'] != emotions or cfg['blocks_per_fold'] != 4:
        raise ValueError(f'Check allocation, emotion order or block count for run {r}')
    ready[r] = {f for f in cfg['holdout_groups'] if missing_file(r, f) is None}
common = sorted(set.intersection(*ready.values()))
selected = common[:N_FOLDS] if FOLD_IDS is None else list(FOLD_IDS)
if (FOLD_IDS is None and len(selected) != N_FOLDS) or not selected or len(set(selected)) != len(selected) or not set(selected) <= set(common):
    for r, cfg in configs.items():
        for f in (FOLD_IDS or cfg['holdout_groups']):
            missing = missing_file(r, f)
            if missing:
                print(f'Run {r}, fold {f}, missing {missing}')
    raise ValueError(f'Insufficient or invalid folds for {PREDICTION_SOURCE}. Completed folds per run: {ready}')
print('Prediction source:', PREDICTION_SOURCE)
print('Selected folds:', selected, '| models per run:', len(selected) * len(BLOCKS))
OUT = ROOT / (PREDICTION_SOURCE + '_allocations_' + '_'.join(map(str, RUNS)) + '_folds_' + '_'.join(map(str, selected)))
OUT.mkdir(exist_ok=True, parents=True)

def load_run(run):
    frames = []
    for fold in selected:
        for block in BLOCKS:
            folder = prediction_dir(run, fold, block)
            metrics = None
            if PREDICTION_SOURCE == 'single_head_retest':
                metrics = json.loads((folder / 'metrics.json').read_text())
                if metrics['tested_final_video_heads'] != 1 or metrics['epoch'] != configs[run]['epochs']:
                    raise ValueError(f'Unexpected model or epoch in {folder}')
            manifest = pd.read_csv(folder.parent / 'test.csv')
            for cond in conditions:
                d = pd.read_csv(folder / f'test_{cond}.csv')
                def trial_keys(frame):
                    trial = frame.video.str.replace(chr(92), '/', regex=False).str.split('/').str[-1]
                    return frame.assign(trial=trial).set_index(['actor','trial'], verify_integrity=True).label.sort_index()
                if (not trial_keys(d).equals(trial_keys(manifest)) or not d.condition.eq(cond).all()
                    or (metrics is not None and len(d) != metrics['conditions'][cond]['total'])
                    or not d[['label','prediction']].isin(range(6)).all().all()):
                    raise ValueError(f'Invalid or incomplete {cond} predictions in {folder}')
                d['trial'] = d.video.str.replace(chr(92), '/', regex=False).str.split('/').str[-1]
                frames.append(d.assign(fold=fold, block=block))
    return configs[run], pd.concat(frames).set_index(keys, verify_integrity=True).sort_index()

runs = {r: load_run(r) for r in RUNS}
cfg0, base = runs[BASELINE]
for r,(cfg,d) in runs.items():
    if not d.index.equals(base.index) or not d.label.equals(base.label):
        raise ValueError(f'Unmatched trials, actors, labels or folds in Run {r}')
    for k in ['classification','epochs','mlp_hidden_dims','seed','lr','batch_size']:
        if cfg[k] != cfg0[k]:
            print(f'Additional difference: Run {r}, {k}: {cfg[k]} vs {cfg0[k]}')
print('Matched test presentations per run:', len(base))


In [ ]:
rows = []
for r, (_, d) in runs.items():
    for (fold,block,c), g in d.groupby(level=['fold','block','condition']):
        counts = pd.crosstab(g.label,g.prediction).reindex(index=range(6), columns=range(6), fill_value=0)
        totals = counts.sum(axis=1)
        if (totals==0).any(): raise ValueError('Missing true emotion in fold')
        for i,true in enumerate(emotions):
            for j,pred in enumerate(emotions):
                rows.append([r,fold,block,c,true,pred,counts.iloc[i,j],totals.iloc[i],100*counts.iloc[i,j]/totals.iloc[i]])
folds = pd.DataFrame(rows,columns=['run','fold','block','condition','true','pred','count','total','percent'])
folds.to_csv(OUT/'fold_confusion_cells.csv',index=False)
wide = folds.pivot(index=['condition','true','pred','fold','block'],columns='run',values='percent')
rows = []
for (c,true,pred), g in wide.groupby(level=['condition','true','pred']):
    for r in comparisons:
        delta = g[r]-g[BASELINE]
        by_fold = g.groupby(level="fold").mean()
        valid = len(by_fold)>1 and (by_fold[r]-by_fold[BASELINE]).std()>1e-12
        t,p = ttest_rel(by_fold[r],by_fold[BASELINE]) if valid else (np.nan,np.nan)
        rows.append([r,c,true,pred,len(delta),g[BASELINE].mean(),g[r].mean(),delta.mean(),delta.std(ddof=1),t,len(by_fold)-1,p])
results = pd.DataFrame(rows,columns=['run','condition','true','pred','n_models','baseline_pct','comparison_pct','delta_pp','std_pp','t','df','p'])
results['allocation'] = results.run.map(RUN_LABELS)
accuracy = results.query('true == pred').copy()
results.to_csv(OUT/'paired_confusion_tests.csv',index=False)
accuracy.to_csv(OUT/'paired_emotion_accuracy.csv',index=False)
with pd.option_context('display.max_rows',None):
    display(accuracy.drop(columns='pred'))


In [ ]:
diagonal = folds[folds.true == folds.pred]
performance = diagonal.groupby(['run','condition','true']).percent.agg(['mean','std'])
table = performance.reset_index().assign(allocation=lambda d: d.run.map(RUN_LABELS))
table.to_csv(OUT/'per_emotion_accuracy_mean_sd.csv', index=False)
display(table.assign(accuracy=lambda d: [f'{m:.1f} ({sd:.1f})' for m, sd in zip(d['mean'], d['std'])])
        .pivot(index=['condition','true'],columns='allocation',values='accuracy'))
fig, grid = plt.subplots(1, len(RUN_LABELS), figsize=(5.7*len(RUN_LABELS),5),
                         sharey=True, squeeze=False, constrained_layout=True)
axes = grid[0]
colors = {'audio_neutral':'#0072B2', 'video_noise':'#D55E00', 'both':'#009E73'}
x = np.arange(len(emotions))
for ax, (run, allocation) in zip(axes, RUN_LABELS.items()):
    for offset, condition in zip(offsets(len(conditions)), conditions):
        g = performance.loc[(run, condition)].reindex(emotions)
        ax.bar(x + offset, g['mean'], yerr=g['std'], width=.6/len(conditions),
               capsize=3, color=colors[condition], label=condition_titles[condition])
    ax.set(xticks=x, xticklabels=emotions, title=allocation, xlabel='Emotion')
    ax.tick_params(axis='x', rotation=45)
axes[0].set_ylabel('Per-emotion accuracy (%, model mean ± SD)')
axes[-1].legend(title='Testing condition', loc='upper left', bbox_to_anchor=(1, 1))
fig.savefig(OUT/'emotion_accuracy_all_runs.png', dpi=180, bbox_inches='tight')
plt.show()


## Audiovisual accuracy gain by emotion
Each point shows the mean within-model gain for one training allocation. Positive values indicate higher audiovisual accuracy. Error bars show ±1 SD across fold/block model differences.


In [ ]:
paired = diagonal.pivot(index=['run', 'fold', 'block', 'true'], columns='condition', values='percent')
if paired[conditions].isna().any().any():
    raise ValueError('Missing condition for a run, fold or emotion.')
for control, label in [('audio_neutral', 'audio only'), ('video_noise', 'video only')]:
    gain = (paired['both'] - paired[control]).groupby(level=['run', 'true']).agg(['mean', 'std'])
    fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
    for offset, (run, allocation) in zip(offsets(len(RUN_LABELS)), RUN_LABELS.items()):
        points = gain.loc[run].reindex(emotions)
        ax.errorbar(np.arange(len(emotions)) + offset, points['mean'], yerr=points['std'],
                    fmt='o', capsize=3, label=allocation)
    ax.axhline(0, color='gray', linewidth=1)
    ax.set(xticks=np.arange(len(emotions)), xticklabels=emotions, xlabel='Emotion',
           ylabel='Accuracy difference (percentage points)', title=f'Audiovisual minus {label}')
    ax.legend(title='Training allocation')
    fig.savefig(OUT / f'audiovisual_minus_{label.replace(" ", "_")}.png', dpi=180)
    plt.show()


In [ ]:
for r in comparisons:
    fig, axes = plt.subplots(1,3,figsize=(18,6),constrained_layout=True)
    sub = results[results.run==r]
    limit = max(1, sub.delta_pp.abs().max())
    for ax,c in zip(axes,conditions):
        cells = sub[sub.condition==c].set_index(['true','pred'])
        grid = cells.delta_pp.unstack().loc[emotions,emotions]
        im=ax.imshow(grid,cmap='RdBu_r',vmin=-limit,vmax=limit)
        for i,e in enumerate(emotions):
            for j,q in enumerate(emotions):
                v,p = cells.loc[(e,q),['delta_pp','p']]
                label = f'p={p:.3g}' if np.isfinite(p) else 'p=n/a'
                ax.text(j,i,f'{v:+.1f}\n{label}',ha='center',va='center',fontsize=8,
                        color='white' if abs(v)>.6*limit else 'black')
        ax.set(xticks=range(6),yticks=range(6),xticklabels=emotions,yticklabels=emotions,
               xlabel='Predicted emotion',ylabel='True emotion',title=condition_titles[c])
        ax.tick_params(axis='x',rotation=45)
    fig.colorbar(im,ax=axes,label='Difference (percentage points)',shrink=.7)
    fig.suptitle(f'{RUN_LABELS[r]} minus {RUN_LABELS[BASELINE]}')
    fig.savefig(OUT/f'confusion_run{r}_minus{BASELINE}.png',dpi=180)
    plt.show()

fig,axes=plt.subplots(1,3,figsize=(17,5),sharey=True,constrained_layout=True)
for ax,c in zip(axes,conditions):
    for offset,r in zip(offsets(len(comparisons)),comparisons):
        g=accuracy[(accuracy.condition==c)&(accuracy.run==r)].set_index('true').loc[emotions]
        x = np.arange(len(emotions)) + offset
        bars = ax.bar(x, g.delta_pp, yerr=g.std_pp, width=.6/len(comparisons), capsize=3,
                           label=f'{RUN_LABELS[r]} − {RUN_LABELS[BASELINE]}')
        significant = g.p.lt(.05).to_numpy()
        ax.scatter(x[significant], g.delta_pp.to_numpy()[significant], marker='*',
                   s=160, color=bars.patches[0].get_facecolor(), edgecolors='black',
                   linewidths=.8, zorder=5)
    ax.axhline(0,color='gray',lw=1)
    ax.set(xticks=range(6),xticklabels=emotions,title=condition_titles[c])
    ax.tick_params(axis='x',rotation=45)
fig.suptitle('★  Paired difference from baseline (exploratory): two-sided p < 0.05 (unadjusted)')
axes[0].set_ylabel('Accuracy difference (percentage points, mean ± paired SD)')
axes[-1].legend()
fig.savefig(OUT/'emotion_accuracy_differences.png',dpi=180)
plt.show()
